# Books Data

In [1]:
# importing libraries
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None)

In [2]:
# paths
INTERIM = Path("../data/interim")

In [5]:
a = pd.read_parquet(INTERIM / "editions.parquet")
b = pd.read_parquet(INTERIM / "editions_extra.parquet")

overlap = set(a.work_id) & set(b.work_id)
print(f"editions   {len(a):,} + {len(b):,}")
print(f"overlapping work_ids: {len(overlap):,}")

editions   1,244,611 + 792,133
overlapping work_ids: 61


In [6]:
a = pd.read_parquet(INTERIM / "english_works.parquet")
b = pd.read_parquet(INTERIM / "english_works_extra.parquet")

works = pd.concat([a, b], ignore_index=True)
dupes = works.work_id.duplicated(keep=False).sum()
print(f"{len(works):,} works, {dupes} duplicate rows")

works = works.sort_values("ratings_total", ascending=False) \
             .drop_duplicates("work_id", keep="first") \
             .reset_index(drop=True)
print(f"{len(works):,} after dedup")

1,002,652 works, 90 duplicate rows
1,002,607 after dedup


In [7]:
for q in ["Dorian Gray", "One Hundred Years of Solitude", "Ivan Ily", "Little Prince"]:
    hits = works[works.title.str.contains(q, case=False, na=False, regex=False)]
    if hits.empty:
        print(f"{q}: NO MATCH")
    else:
        r = hits.nlargest(1, "ratings_total").iloc[0]
        print(f"{q}: {r.title} ({r.ratings_total:,})")

Dorian Gray: The Picture of Dorian Gray (683,950)
One Hundred Years of Solitude: One Hundred Years of Solitude (578,926)
Ivan Ily: The Death of Ivan Ilych (55,051)
Little Prince: The Little Prince (878,752)


In [9]:
works = works.drop(columns=["genre"])

In [10]:
print("only in a:", set(a.columns) - set(b.columns))
print("only in b:", set(b.columns) - set(a.columns))

only in a: {'rating_weighted'}
only in b: {'avg_rating', '_has_desc', '_is_en'}


In [11]:
b = b.rename(columns={"avg_rating": "rating_weighted"})
b = b.drop(columns=["_has_desc", "_is_en"])

works = pd.concat([a, b], ignore_index=True) \
          .sort_values("ratings_total", ascending=False) \
          .drop_duplicates("work_id", keep="first") \
          .reset_index(drop=True)

if "genre" in works.columns:
    works = works.drop(columns=["genre"])

print(set(a.columns) ^ set(b.columns) or "columns match")
print(f"{len(works):,} works")

columns match
1,002,607 works


In [12]:
out = INTERIM / "english_works_all.parquet"
works.to_parquet(out, index=False)

print(f"{len(works):,} works -> {out}  ({out.stat().st_size / 1e6:.0f} MB)")

check = pd.read_parquet(out)
assert len(check) == len(works)
assert check.work_id.is_unique
print("verified")

1,002,607 works -> ../data/interim/english_works_all.parquet  (1561 MB)
verified


## Books File Analysis

In [6]:
books = pd.read_parquet(
    INTERIM / "english_works_all.parquet",
    columns=["work_id", "title", "authors", "description", "genres",
             "tags", "tag_counts", "book_id_all", "series", "similar_books",
             "ratings_total", "reviews_total", "rating_weighted",
             "year_first", "pages_median", "editions"],
)

In [7]:
books.head()

,work_id,title,authors,description,genres,tags,tag_counts,book_id_all,series,similar_books,ratings_total,reviews_total,rating_weighted,year_first,pages_median,editions
0,2792775,"The Hunger Games (The Hunger Games, #1)","[{'author_id': '153394', 'role': ''}]",Winning will make you famous.\nLosing means ce...,[young_adult],"[young-adult, fiction, dystopian, dystopia, fa...","[3315957, 1766978, 1659179, 1416250, 1385816, ...","[10197660, 10255274, 10282880, 10327392, 10616...",[339872],"[1902241, 146499, 954674, 9917938, 10165727, 7...",5064668,156478,4.34,2008.0,384.0,126
1,4640799,Harry Potter and the Sorcerer's Stone (Harry P...,"[{'author_id': '1077326', 'role': ''}, {'autho...",Harry Potter's life is miserable. His parents ...,[fantasy_paranormal],"[fantasy, young-adult, fiction, harry-potter, ...","[11847821, 3718993, 3316936, 2144163, 1311521,...","[10180184, 1033918, 10385172, 10435027, 110197...",[167817],"[13830, 127586, 121822, 37586, 6164358, 807968...",4970387,78500,4.45,1997.0,285.0,244
2,3212258,"Twilight (Twilight, #1)","[{'author_id': '941441', 'role': ''}]",About three things I was absolutely positive.\...,[fantasy_paranormal],"[young-adult, fantasy, vampires, ya, fiction, ...","[1529970, 1334012, 689122, 611516, 586966, 483...","[10244455, 1029083, 1035885, 10771500, 1111273...",[165622],"[1326258, 140077, 35729, 30123413, 11260526, 9...",3991256,95594,3.57,2005.0,476.0,92
3,3275794,To Kill a Mockingbird,"[{'author_id': '1825', 'role': ''}]",The unforgettable novel of a childhood in a sl...,[history_biography],"[classics, classic, historical-fiction, clàssi...","[6524280, 1344924, 1026960, 762264, 640710, 53...","[10257528, 10370664, 10623852, 10754520, 11192...",[735877],"[1934, 2156, 15638, 53835, 77142, 5114, 116020...",3399207,73801,4.26,1960.0,331.0,210
4,245494,The Great Gatsby,"[{'author_id': '3190', 'role': ''}]","THE GREAT GATSBY, F. Scott Fitzgerald's third ...","[biography, fiction, historical fiction, histo...","[classics, fiction, classic, literature, schoo...","[10839416, 5134512, 1884073, 903665, 887568, 8...","[10125603, 10194387, 10301147, 10304725, 10381...",[],"[48203, 337113, 176972, 188087, 10956, 12722, ...",2841340,52125,3.89,1925.0,189.5,328


In [8]:
books.shape

(1002607, 16)

In [9]:
books.shape
books['work_id'].nunique()
if books.shape[0] == books['work_id'].nunique():
    print('work id is primary key.')
else:
    print('check for duplicates')

work id is primary key.


In [10]:
books.columns

Index(['work_id', 'title', 'authors', 'description', 'genres', 'tags',
       'tag_counts', 'book_id_all', 'series', 'similar_books', 'ratings_total',
       'reviews_total', 'rating_weighted', 'year_first', 'pages_median',
       'editions'],
      dtype='str')

In [11]:
titles = ["Little Prince", "Picture of Dorian Gray"]

for q in titles:
    hits = books[books.title.str.contains(q, case=False, na=False, regex=False)]
    if hits.empty:
        print(f"=== {q}: NO MATCH\n")
        continue

    r = hits.nlargest(1, "ratings_total").iloc[0]

    print(f"=== {r.title}")
    print(f"rating   {r.rating_weighted:.2f}   ratings {r.ratings_total:,}   reviews {r.reviews_total:,}")
    print(f"genres   {list(r.genres)}")
    print(f"\ndescription:\n{r.description}")
    print(f"\ntags ({len(r.tags)}):")
    for t, c in zip(r.tags, r.tag_counts):
        print(f"  {c:>10,}  {t}")
    print("\n" + "-" * 70 + "\n")

=== The Little Prince
rating   4.28   ratings 878,752   reviews 24,880
genres   ['children']

description:
Moral allegory and spiritual autobiography, The Little Prince is the most translated book in the French language. With a timeless charm it tells the story of a little boy who leaves the safety of his own tiny planet to travel the universe, learning the vagaries of adult behaviour through a series of extraordinary encounters. His personal odyssey culminates in a voyage to Earth and further adventures.

tags (77):
   3,833,138  classics
   2,121,014  fiction
   1,531,321  fantasy
   1,092,573  childrens
     897,490  children
     749,796  classic
     636,833  french
     487,124  children-s
     448,546  young-adult
     429,403  philosophy
     385,261  children-s-books
     300,490  childhood
     296,692  literature
     279,183  kids
     237,639  childrens-books
     213,294  french-literature
     197,806  school
     181,090  1001-books
     174,357  france
     150,046  fa

In [12]:
gm = pd.read_parquet("../data/raw/book_genres.parquet")
gm_dict = dict(zip(gm.book_id, gm.genres))

def map_genres(book_ids):
    out = set()
    for b in book_ids:
        out.update(gm_dict.get(str(b), []))
    return sorted(out)

books["genres_map"] = books.book_id_all.apply(map_genres)

print(f"books with mapped genres: {books.genres_map.apply(len).gt(0).mean():.1%}")
print(books.genres_map.explode().value_counts().to_string())

books with mapped genres: 84.2%
genres_map
fiction               457763
romance               272291
non-fiction           258174
biography             255241
historical fiction    255241
history               255241
fantasy               212175
paranormal            212175
crime                 183081
mystery               183081
thriller              183081
young-adult           125700
children              123578
comics                 72597
graphic                72597
poetry                 36905


In [13]:
lp = books[books.title.str.contains("Little Prince", case=False, na=False, regex=False)] \
        .nlargest(1, "ratings_total").iloc[0]

print("from files:", list(lp.genres))
print("from map  :", list(lp.genres_map))

from files: ['children']
from map  : ['children', 'fantasy', 'fiction', 'paranormal', 'young-adult']


In [14]:
books["genres"] = [
    list(m) if len(m) else list(g)
    for m, g in zip(books.genres_map, books.genres)
]
books = books.drop(columns=["genres_map"])

print(books.genres.explode().value_counts().to_string())
print(f"\nno genre: {books.genres.apply(len).eq(0).mean():.1%}")

genres
fiction               457763
romance               272291
non-fiction           258174
biography             255241
historical fiction    255241
history               255241
fantasy               212175
paranormal            212175
crime                 183081
mystery               183081
thriller              183081
young-adult           125700
children              123578
comics                 72597
graphic                72597
poetry                 36905

no genre: 15.8%


In [15]:
books.head()

,work_id,title,authors,description,genres,tags,tag_counts,book_id_all,series,similar_books,ratings_total,reviews_total,rating_weighted,year_first,pages_median,editions
0,2792775,"The Hunger Games (The Hunger Games, #1)","[{'author_id': '153394', 'role': ''}]",Winning will make you famous.\nLosing means ce...,"[crime, fantasy, fiction, mystery, paranormal,...","[young-adult, fiction, dystopian, dystopia, fa...","[3315957, 1766978, 1659179, 1416250, 1385816, ...","[10197660, 10255274, 10282880, 10327392, 10616...",[339872],"[1902241, 146499, 954674, 9917938, 10165727, 7...",5064668,156478,4.34,2008.0,384.0,126
1,4640799,Harry Potter and the Sorcerer's Stone (Harry P...,"[{'author_id': '1077326', 'role': ''}, {'autho...",Harry Potter's life is miserable. His parents ...,"[children, crime, fantasy, fiction, mystery, p...","[fantasy, young-adult, fiction, harry-potter, ...","[11847821, 3718993, 3316936, 2144163, 1311521,...","[10180184, 1033918, 10385172, 10435027, 110197...",[167817],"[13830, 127586, 121822, 37586, 6164358, 807968...",4970387,78500,4.45,1997.0,285.0,244
2,3212258,"Twilight (Twilight, #1)","[{'author_id': '941441', 'role': ''}]",About three things I was absolutely positive.\...,"[fantasy, fiction, paranormal, romance, young-...","[young-adult, fantasy, vampires, ya, fiction, ...","[1529970, 1334012, 689122, 611516, 586966, 483...","[10244455, 1029083, 1035885, 10771500, 1111273...",[165622],"[1326258, 140077, 35729, 30123413, 11260526, 9...",3991256,95594,3.57,2005.0,476.0,92
3,3275794,To Kill a Mockingbird,"[{'author_id': '1825', 'role': ''}]",The unforgettable novel of a childhood in a sl...,"[biography, crime, fiction, historical fiction...","[classics, classic, historical-fiction, clàssi...","[6524280, 1344924, 1026960, 762264, 640710, 53...","[10257528, 10370664, 10623852, 10754520, 11192...",[735877],"[1934, 2156, 15638, 53835, 77142, 5114, 116020...",3399207,73801,4.26,1960.0,331.0,210
4,245494,The Great Gatsby,"[{'author_id': '3190', 'role': ''}]","THE GREAT GATSBY, F. Scott Fitzgerald's third ...","[biography, fiction, historical fiction, histo...","[classics, fiction, classic, literature, schoo...","[10839416, 5134512, 1884073, 903665, 887568, 8...","[10125603, 10194387, 10301147, 10304725, 10381...",[],"[48203, 337113, 176972, 188087, 10956, 12722, ...",2841340,52125,3.89,1925.0,189.5,328


In [16]:
titles = ["Little Prince", "Picture of Dorian Gray", "One Hundred Years of Solitude",
          "Ivan Ily", "Hitchhiker"]

for q in titles:
    hits = books[books.title.str.contains(q, case=False, na=False, regex=False)]
    if hits.empty:
        print(f"=== {q}: NO MATCH\n")
        continue

    r = hits.nlargest(1, "ratings_total").iloc[0]

    print(f"=== {r.title}")
    print(f"rating   {r.rating_weighted:.2f}   ratings {r.ratings_total:,}   reviews {r.reviews_total:,}")
    print(f"genres   {list(r.genres)}")
    print(f"\ndescription:\n{r.description}")
    print(f"\ntags ({len(r.tags)}):")
    for t, c in zip(r.tags, r.tag_counts):
        print(f"  {c:>10,}  {t}")
    print("\n" + "-" * 70 + "\n")

=== The Little Prince
rating   4.28   ratings 878,752   reviews 24,880
genres   ['children', 'fantasy', 'fiction', 'paranormal', 'young-adult']

description:
Moral allegory and spiritual autobiography, The Little Prince is the most translated book in the French language. With a timeless charm it tells the story of a little boy who leaves the safety of his own tiny planet to travel the universe, learning the vagaries of adult behaviour through a series of extraordinary encounters. His personal odyssey culminates in a voyage to Earth and further adventures.

tags (77):
   3,833,138  classics
   2,121,014  fiction
   1,531,321  fantasy
   1,092,573  childrens
     897,490  children
     749,796  classic
     636,833  french
     487,124  children-s
     448,546  young-adult
     429,403  philosophy
     385,261  children-s-books
     300,490  childhood
     296,692  literature
     279,183  kids
     237,639  childrens-books
     213,294  french-literature
     197,806  school
     181,09

## Testing signal from description

tags might give some value, but they are too much to clean. if description can extract similar tags, they would not be needed.

In [17]:
from transformers import pipeline

clf = pipeline("text-classification",
               model="j-hartmann/emotion-english-distilroberta-base",
               top_k=None, truncation=True)

for q in ["Little Prince", "Ivan Ily", "Hitchhiker", "Dorian Gray"]:
    r = books[books.title.str.contains(q, case=False, na=False, regex=False)] \
            .nlargest(1, "ratings_total").iloc[0]
    scores = {d["label"]: round(d["score"], 3) for d in clf(r.description)[0]}
    print(f"{q:<16} {scores}")

/Users/navyajain/Downloads/projects/book-recommendation-engine/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 18799.71it/s]


Little Prince    {'joy': 0.682, 'neutral': 0.262, 'fear': 0.014, 'surprise': 0.014, 'sadness': 0.013, 'disgust': 0.011, 'anger': 0.003}
Ivan Ily         {'surprise': 0.765, 'fear': 0.196, 'neutral': 0.018, 'sadness': 0.007, 'disgust': 0.005, 'anger': 0.005, 'joy': 0.003}
Hitchhiker       {'neutral': 0.869, 'surprise': 0.031, 'disgust': 0.03, 'anger': 0.024, 'joy': 0.023, 'fear': 0.016, 'sadness': 0.007}
Dorian Gray      {'disgust': 0.625, 'fear': 0.329, 'anger': 0.016, 'neutral': 0.013, 'sadness': 0.009, 'surprise': 0.006, 'joy': 0.003}


In [19]:
import requests

BASE = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads"

r = requests.head(f"{BASE}/goodreads_interactions.csv", allow_redirects=True)
print(f"{int(r.headers['Content-Length']) / 1e9:.1f} GB")

4.3 GB


In [ ]:
books[["work_id", "book_id_all"]].to_parquet("../data/interim/catalog_ids.parquet", index=False)